# Vibe Sewing Lab

**AI-powered Creative Sewing Assistant**

Vibe Sewing Lab is an intelligent creative assistant for artisan sewing professionals, especially makers who design handmade bags.

It does not replace human creativity.

It amplifies it.

## Features

- Idea generation for handmade bag concepts
- Image-based design analysis
- Technical sketch generation
- Material planning
- Cost estimation and price suggestion
- Creative storytelling
- Marketing copy for multiple channels
- Technical assistant with RAG
- Project gallery and analytics dashboard

## Tech Stack

- Python
- Google Colab
- Gradio
- OpenCV
- Pillow
- NumPy
- Pandas
- Plotly
- FAISS
- Sentence Transformers
- LangChain
- Ollama
- SQLite

### Celula 1 - Instalacao

In [ ]:
# Dependencias para Colab + execucao local com LLM real (Qwen/Ollama/API)
%pip install -q gradio opencv-python pillow numpy pandas matplotlib plotly networkx scikit-learn faiss-cpu sentence-transformers transformers accelerate bitsandbytes pyyaml requests nbformat nbconvert torch
%pip install -q google-generativeai openai pypdf ollama langchain langchain-community langchain-text-splitters

### Celula 2 - Estrutura de pastas

In [ ]:
from pathlib import Path
import os


def _default_project_root() -> Path:
    if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ:
        return Path("/content/vibe-sewing-lab")
    return Path.cwd() / "vibe-sewing-lab"


PROJECT_ROOT = _default_project_root()
DIRS = [
    "src/core",
    "src/application",
    "src/domain",
    "src/infrastructure/llm",
    "src/infrastructure/vision",
    "src/infrastructure/rag",
    "src/infrastructure/persistence",
    "src/infrastructure/utils",
    "src/ui",
    "src/analytics",
    "data/knowledge_base",
    "data/projects",
    "data/exports",
    "configs",
    "assets/sample_images",
    "docs",
    "tests",
]

for directory in DIRS:
    (PROJECT_ROOT / directory).mkdir(parents=True, exist_ok=True)

for init_file in [
    PROJECT_ROOT / "src" / "__init__.py",
    PROJECT_ROOT / "src" / "core" / "__init__.py",
    PROJECT_ROOT / "src" / "application" / "__init__.py",
    PROJECT_ROOT / "src" / "domain" / "__init__.py",
]:
    init_file.touch(exist_ok=True)

print(f"Projeto criado em: {PROJECT_ROOT}")

### Celula 3 - Modelos centrais

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field, asdict
from enum import Enum
from typing import Any, Dict, List, Optional
from uuid import uuid4
import json


class DifficultyLevel(str, Enum):
    EASY = "easy"
    MEDIUM = "medium"
    HARD = "hard"


@dataclass
class MaterialItem:
    name: str
    category: str
    quantity: float
    unit: str
    estimated_unit_cost: float
    notes: str = ""

    @property
    def subtotal(self) -> float:
        return self.quantity * self.estimated_unit_cost


@dataclass
class CostBreakdown:
    materials_cost: float
    labor_hours: float
    labor_hour_rate: float
    overhead_percentage: float
    margin_percentage: float

    @property
    def labor_cost(self) -> float:
        return self.labor_hours * self.labor_hour_rate

    @property
    def overhead_cost(self) -> float:
        return (self.materials_cost + self.labor_cost) * (self.overhead_percentage / 100)

    @property
    def base_cost(self) -> float:
        return self.materials_cost + self.labor_cost + self.overhead_cost

    @property
    def suggested_price(self) -> float:
        return self.base_cost * (1 + self.margin_percentage / 100)


@dataclass
class SewingProject:
    title: str
    description: str
    source_type: str
    id: str = field(default_factory=lambda: str(uuid4()))
    concepts: List[Dict[str, Any]] = field(default_factory=list)
    image_analysis: Dict[str, Any] = field(default_factory=dict)
    materials: List[MaterialItem] = field(default_factory=list)
    costing: Dict[str, Any] = field(default_factory=dict)
    creative_assets: Dict[str, Any] = field(default_factory=dict)
    production_plan: Dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> Dict[str, Any]:
        return {
            "id": self.id,
            "title": self.title,
            "description": self.description,
            "source_type": self.source_type,
            "concepts": self.concepts,
            "image_analysis": self.image_analysis,
            "materials": [asdict(item) for item in self.materials],
            "costing": self.costing,
            "creative_assets": self.creative_assets,
            "production_plan": self.production_plan,
        }

    def to_json(self) -> str:
        return json.dumps(self.to_dict(), ensure_ascii=False, indent=2)


### Celula 4 - Abstracao de providers LLM

In [ ]:
from abc import ABC, abstractmethod
from typing import Optional
import os


class LLMProviderError(RuntimeError):
    pass


def _is_colab_runtime() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ


class BaseLLMProvider(ABC):
    @abstractmethod
    def generate(self, prompt: str) -> str:
        raise NotImplementedError


class GeminiLLMProvider(BaseLLMProvider):
    def __init__(self, api_key: Optional[str] = None, model_name: str = "gemini-1.5-flash"):
        self.api_key = api_key or os.getenv("GEMINI_API_KEY")
        self.model_name = model_name
        self._model = None

    def _lazy_init(self):
        if not self.api_key:
            raise LLMProviderError("GEMINI_API_KEY nao definido.")
        if self._model is None:
            import google.generativeai as genai
            genai.configure(api_key=self.api_key)
            self._model = genai.GenerativeModel(self.model_name)

    def generate(self, prompt: str) -> str:
        self._lazy_init()
        response = self._model.generate_content(prompt)
        text = getattr(response, "text", "")
        if not text:
            raise LLMProviderError("Gemini retornou resposta vazia.")
        return text


class OpenAILLMProvider(BaseLLMProvider):
    def __init__(self, api_key: Optional[str] = None, model_name: str = "gpt-4o-mini"):
        self.api_key = api_key or os.getenv("OPENAI_API_KEY")
        self.model_name = model_name

    def generate(self, prompt: str) -> str:
        if not self.api_key:
            raise LLMProviderError("OPENAI_API_KEY nao definido.")
        from openai import OpenAI
        client = OpenAI(api_key=self.api_key)
        response = client.responses.create(model=self.model_name, input=prompt)
        text = getattr(response, "output_text", "")
        if not text:
            raise LLMProviderError("OpenAI retornou resposta vazia.")
        return text


class OllamaLLMProvider(BaseLLMProvider):
    def __init__(self, model_name: str = "qwen2.5:3b-instruct", host: Optional[str] = None):
        self.model_name = model_name
        self.host = host or os.getenv("OLLAMA_HOST")
        self._client = None

    def _lazy_client(self):
        if self._client is None:
            from ollama import Client
            self._client = Client(host=self.host) if self.host else Client()
        return self._client

    def generate(self, prompt: str) -> str:
        client = self._lazy_client()
        response = client.chat(
            model=self.model_name,
            messages=[
                {"role": "system", "content": "Voce e um especialista em design e confeccao de bolsas artesanais."},
                {"role": "user", "content": prompt},
            ],
            options={"temperature": 0.7},
        )
        text = response.get("message", {}).get("content", "")
        if not text:
            raise LLMProviderError("Ollama retornou resposta vazia. Verifique se o modelo foi baixado.")
        return text


class LocalTransformersLLMProvider(BaseLLMProvider):
    def __init__(self, model_name: str = "Qwen/Qwen2.5-3B-Instruct"):
        self.model_name = model_name
        self._pipeline = None

    def _lazy_init(self):
        if self._pipeline is None:
            from transformers import pipeline
            self._pipeline = pipeline(
                "text-generation",
                model=self.model_name,
                torch_dtype="auto",
                device_map="auto",
            )

    def generate(self, prompt: str) -> str:
        self._lazy_init()
        result = self._pipeline(
            prompt,
            max_new_tokens=420,
            do_sample=True,
            temperature=0.7,
            top_p=0.92,
            repetition_penalty=1.05,
        )
        if isinstance(result, list) and result:
            text = str(result[0].get("generated_text", ""))
            if text:
                return text
        raise LLMProviderError("Transformers retornou resposta vazia.")


def get_default_llm_provider(provider_name: str = "auto", model_name: Optional[str] = None) -> BaseLLMProvider:
    provider = (provider_name or "auto").lower()

    if provider == "gemini":
        return GeminiLLMProvider(model_name=model_name or "gemini-1.5-flash")
    if provider == "openai":
        return OpenAILLMProvider(model_name=model_name or "gpt-4o-mini")
    if provider == "ollama":
        return OllamaLLMProvider(model_name=model_name or "qwen2.5:3b-instruct")
    if provider in {"local", "transformers", "qwen"}:
        return LocalTransformersLLMProvider(model_name=model_name or "Qwen/Qwen2.5-3B-Instruct")

    if _is_colab_runtime():
        return LocalTransformersLLMProvider(model_name=model_name or "Qwen/Qwen2.5-3B-Instruct")
    return OllamaLLMProvider(model_name=model_name or "qwen2.5:3b-instruct")

### Celula 5 - Servico de ideacao

In [ ]:
from typing import Dict, List
import json
import re


class IdeaGeneratorService:
    def __init__(self, llm_provider: BaseLLMProvider):
        self.llm_provider = llm_provider

    def _fallback_concepts(self, user_prompt: str, count: int):
        prompt = re.sub(r"\s+", " ", user_prompt).strip()
        prompt_summary = (prompt[:90] + "...") if len(prompt) > 90 else prompt
        if not prompt_summary:
            prompt_summary = "handmade bag project"

        concepts = []
        for index in range(count):
            concepts.append(
                {
                    "name": f"Concept {index + 1}",
                    "target_audience": prompt_summary,
                    "aesthetic": "artisan contemporary",
                    "suggested_materials": "Main fabric, lining, interfacing, zipper, reinforced handles",
                    "differential": "Balanced handmade look with practical organization",
                    "difficulty": "medium",
                }
            )
        return concepts

    def _extract_json_array(self, text: str):
        try:
            parsed = json.loads(text)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass

        match = re.search(r"\[.*\]", text, re.DOTALL)
        if not match:
            return None

        try:
            parsed = json.loads(match.group(0))
            if isinstance(parsed, list):
                return parsed
        except Exception:
            return None
        return None

    def generate_concepts(self, user_prompt: str, count: int = 10) -> List[Dict[str, str]]:
        prompt = f"""
Voce e um diretor criativo especialista em bolsas artesanais.
Gere {count} conceitos originais para a ideia abaixo.

Ideia da cliente:
{user_prompt}

Responda em JSON puro com uma lista de objetos contendo exatamente:
name, target_audience, aesthetic, suggested_materials, differential, difficulty

difficulty deve ser: easy, medium ou hard.
"""
        try:
            raw = self.llm_provider.generate(prompt)
            parsed = self._extract_json_array(raw)
        except Exception:
            parsed = None

        if not isinstance(parsed, list) or not parsed:
            return self._fallback_concepts(user_prompt, count)

        concepts: List[Dict[str, str]] = []
        for item in parsed[:count]:
            if not isinstance(item, dict):
                continue
            concepts.append({
                "name": str(item.get("name", "")).strip(),
                "target_audience": str(item.get("target_audience", "")).strip(),
                "aesthetic": str(item.get("aesthetic", "")).strip(),
                "suggested_materials": str(item.get("suggested_materials", "")).strip(),
                "differential": str(item.get("differential", "")).strip(),
                "difficulty": str(item.get("difficulty", "medium")).lower().strip(),
            })

        concepts = [c for c in concepts if c["name"] and c["target_audience"]]
        if not concepts:
            return self._fallback_concepts(user_prompt, count)
        return concepts

### Celula 6 - Analise de imagem e Computer Vision

In [ ]:
import cv2
import numpy as np
from PIL import Image
from typing import Any, Dict


class ImageAnalysisService:
    def analyze(self, image: Image.Image) -> Dict[str, Any]:
        image_np = np.array(image.convert("RGB"))
        gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
        edges = cv2.Canny(gray, 80, 180)

        mean_color = image_np.mean(axis=(0, 1)).tolist()
        edge_density = float((edges > 0).mean())

        difficulty = "easy"
        if edge_density > 0.08:
            difficulty = "medium"
        if edge_density > 0.15:
            difficulty = "hard"

        return {
            "style": "artesanal contemporâneo",
            "format": "retangular estruturada",
            "structure": "semi-rígida",
            "type": "bolsa utilitária",
            "difficulty": difficulty,
            "dominant_rgb": [round(c, 2) for c in mean_color],
            "edge_density": round(edge_density, 4),
            "detected_components": [
                "alças",
                "bolsos",
                "costuras aparentes"
            ],
            "textures": [
                "tecido encorpado"
            ]
        }


### Celula 7 - Sketch generator simples

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch


class SketchGeneratorService:
    def generate_sketch_views(self, title: str):
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        views = ["Frontal", "Lateral", "Superior"]
        for ax, view in zip(axes, views):
            ax.set_title(f"{title} - Vista {view}")
            body = FancyBboxPatch(
                (0.2, 0.25), 0.6, 0.45,
                boxstyle="round,pad=0.02,rounding_size=0.03",
                fill=False,
                linewidth=2
            )
            handle = Rectangle((0.38, 0.72), 0.24, 0.08, fill=False, linewidth=2)
            ax.add_patch(body)
            ax.add_patch(handle)
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.axis("off")

        plt.tight_layout()
        return fig


### Celula 8 - Lista de materiais

In [ ]:
from typing import List


class MaterialsService:
    def suggest_materials(self, project: SewingProject) -> List[MaterialItem]:
        analysis = project.image_analysis or {}
        difficulty = analysis.get("difficulty", "medium")

        base_items = [
            MaterialItem("Tecido principal", "tecido", 1.5, "m", 32.0, "Material externo"),
            MaterialItem("Forro", "forro", 1.2, "m", 18.0, "Parte interna"),
            MaterialItem("Manta estruturante", "estrutura", 1.0, "m", 22.0, "Sustentação"),
            MaterialItem("Zíper metálico", "ferragem", 2.0, "un", 8.5, "Fechamento principal"),
            MaterialItem("Alça", "componente", 1.0, "par", 24.0, "Alças reforçadas"),
            MaterialItem("Linha premium", "aviamento", 2.0, "un", 6.0, "Costura externa e interna"),
        ]

        if difficulty == "hard":
            base_items.append(
                MaterialItem("Divisória interna", "estrutura", 1.0, "un", 14.0, "Organização interna")
            )

        return base_items


### Celula 9 - Custos e precificacao

In [ ]:
class CostService:
    def calculate(
        self,
        materials: List[MaterialItem],
        labor_hours: float = 6.0,
        labor_hour_rate: float = 25.0,
        overhead_percentage: float = 12.0,
        margin_percentage: float = 80.0
    ) -> Dict[str, float]:
        materials_cost = sum(item.subtotal for item in materials)

        cost = CostBreakdown(
            materials_cost=materials_cost,
            labor_hours=labor_hours,
            labor_hour_rate=labor_hour_rate,
            overhead_percentage=overhead_percentage,
            margin_percentage=margin_percentage,
        )

        return {
            "materials_cost": round(cost.materials_cost, 2),
            "labor_cost": round(cost.labor_cost, 2),
            "overhead_cost": round(cost.overhead_cost, 2),
            "base_cost": round(cost.base_cost, 2),
            "suggested_price": round(cost.suggested_price, 2),
            "margin_percentage": margin_percentage,
        }


### Celula 10 - IA criativa e marketing

In [ ]:
class CreativeStudioService:
    def __init__(self, llm_provider: BaseLLMProvider):
        self.llm_provider = llm_provider

    def _fallback_brand_assets(self, project: SewingProject) -> Dict[str, str]:
        title = (project.title or "Vibe Sewing Collection").strip()
        description = (project.description or "Handmade bag concept.").strip()
        analysis = project.image_analysis or {}
        difficulty = str(analysis.get("difficulty", "medium")).strip().lower()

        story = (
            f"{title} is a handmade bag concept designed to combine craftsmanship, functionality, and a clear visual identity. "
            f"The project description highlights: {description}"
        )

        return {
            "collection_name": title,
            "story": story,
            "description": description,
            "slogan": "Handmade with character.",
            "instagram": f"Introducing {title}: a {difficulty} handmade bag concept built for everyday use and a strong artisan presence.",
            "linkedin": f"{title} showcases a structured handmade product concept with clear design intent and production readiness.",
            "facebook": f"Meet {title}, a new handmade bag idea focused on utility, style, and craft.",
            "pinterest": f"{title} moodboard: artisan textures, functional structure, and contemporary handmade bag styling.",
            "etsy": f"{title} is a handmade bag concept with thoughtful materials, practical construction, and an artisan finish.",
            "shopee": f"{title} brings a polished handmade aesthetic with practical everyday performance.",
            "mercado_livre": f"{title} is positioned as a handmade bag product for customers who value craft, structure, and uniqueness.",
            "image_prompt": f"Studio product shot of {title}, handmade bag concept, artisan contemporary styling, neutral background, detailed stitching, realistic materials.",
            "video_prompt": f"Short promotional video of {title}, showing the handmade bag from multiple angles, stitching details, hardware, and practical use cases.",
        }

    def _extract_json_object(self, text: str):
        import json
        import re

        try:
            parsed = json.loads(text)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            pass

        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            return None

        try:
            parsed = json.loads(match.group(0))
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            return None
        return None

    def generate_brand_assets(self, project: SewingProject) -> Dict[str, str]:
        prompt = f"""
Crie ativos criativos para uma bolsa artesanal com base nas informacoes abaixo.

Titulo: {project.title}
Descricao: {project.description}
Analise: {project.image_analysis}

Responda em JSON puro com as chaves:
collection_name, story, description, slogan, instagram, linkedin, facebook, pinterest, etsy, shopee, mercado_livre, image_prompt, video_prompt

Todos os campos devem ser preenchidos e coerentes com o produto.
"""
        try:
            raw = self.llm_provider.generate(prompt)
            parsed = self._extract_json_object(raw)
        except Exception:
            parsed = None

        if not isinstance(parsed, dict):
            return self._fallback_brand_assets(project)

        required = [
            "collection_name", "story", "description", "slogan", "instagram", "linkedin",
            "facebook", "pinterest", "etsy", "shopee", "mercado_livre",
            "image_prompt", "video_prompt",
        ]

        missing = [k for k in required if not str(parsed.get(k, "")).strip()]
        if missing:
            return self._fallback_brand_assets(project)

        return {k: str(parsed[k]).strip() for k in required}

### Celula 11 - RAG basico com FAISS

In [ ]:
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
import numpy as np


class SimpleRAGService:
    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        self.texts = []
        self.index = None
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=80,
            separators=["\n\n", "\n", ". ", " ", ""],
        )

    def _add_embeddings(self, texts):
        if not texts:
            return
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        self.texts.extend(texts)

        if self.index is None:
            self.index = faiss.IndexFlatL2(embeddings.shape[1])

        self.index.add(embeddings.astype("float32"))

    def ingest_texts(self, texts):
        if not texts:
            return
        normalized = [t.strip() for t in texts if isinstance(t, str) and t.strip()]
        self._add_embeddings(normalized)

    def ingest_long_text(self, text: str):
        if not text or not text.strip():
            raise ValueError("Texto para base tecnica esta vazio.")
        chunks = self.splitter.split_text(text)
        chunks = [chunk.strip() for chunk in chunks if chunk.strip()]
        if not chunks:
            raise ValueError("Nao foi possivel gerar chunks a partir do texto informado.")
        self._add_embeddings(chunks)

    def query(self, question: str, top_k: int = 3):
        if self.index is None or not self.texts:
            return []
        if not question or not question.strip():
            return []

        question_embedding = self.model.encode([question], convert_to_numpy=True).astype("float32")
        _, indices = self.index.search(question_embedding, min(top_k, len(self.texts)))
        return [self.texts[i] for i in indices[0] if i < len(self.texts)]

    def answer_with_context(self, question: str, llm_provider: BaseLLMProvider, top_k: int = 3) -> Dict[str, Any]:
        contexts = self.query(question, top_k=top_k)
        if not contexts:
            raise ValueError("Base de conhecimento vazia. Adicione texto tecnico antes de consultar.")

        prompt = f"""
Voce e um assistente tecnico de confeccao de bolsas artesanais.
Use apenas o contexto abaixo para responder de forma objetiva e acionavel.

Contexto:
{chr(10).join([f'- {c}' for c in contexts])}

Pergunta: {question}

Formato esperado:
- Resumo tecnico
- Passo a passo
- Cuidados de qualidade
"""
        answer = llm_provider.generate(prompt)
        return {
            "answer": answer.strip(),
            "contexts": contexts,
        }

Celula 12 - Persistencia local

In [ ]:
import sqlite3
from pathlib import Path


class SQLiteProjectRepository:
    def __init__(self, db_path: Path):
        self.db_path = db_path
        self._initialize()

    def _initialize(self):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS projects (
                id TEXT PRIMARY KEY,
                title TEXT NOT NULL,
                description TEXT NOT NULL,
                source_type TEXT NOT NULL,
                payload TEXT NOT NULL
            )
        """)
        conn.commit()
        conn.close()

    def save(self, project: SewingProject):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("""
            INSERT OR REPLACE INTO projects (id, title, description, source_type, payload)
            VALUES (?, ?, ?, ?, ?)
        """, (
            project.id,
            project.title,
            project.description,
            project.source_type,
            project.to_json()
        ))
        conn.commit()
        conn.close()

    def list_projects(self):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT id, title, description, source_type FROM projects ORDER BY rowid DESC")
        rows = cursor.fetchall()
        conn.close()
        return rows


### Celula 13 - Orquestrador principal

In [ ]:
class VibeSewingOrchestrator:
    def __init__(
        self,
        llm_provider: BaseLLMProvider,
        repository: SQLiteProjectRepository
    ):
        self.idea_service = IdeaGeneratorService(llm_provider)
        self.image_service = ImageAnalysisService()
        self.sketch_service = SketchGeneratorService()
        self.materials_service = MaterialsService()
        self.cost_service = CostService()
        self.creative_service = CreativeStudioService(llm_provider)
        self.repository = repository

    def create_project_from_text(self, title: str, description: str) -> SewingProject:
        project = SewingProject(
            title=title,
            description=description,
            source_type="text"
        )
        project.concepts = self.idea_service.generate_concepts(description)
        project.materials = self.materials_service.suggest_materials(project)
        project.costing = self.cost_service.calculate(project.materials)
        project.creative_assets = self.creative_service.generate_brand_assets(project)
        project.production_plan = {
            "checklist": [
                "Definir modelagem",
                "Separar materiais",
                "Cortar tecido externo",
                "Cortar forro",
                "Aplicar estrutura",
                "Costurar bolsos",
                "Montar corpo",
                "Instalar ferragens",
                "Revisar acabamento",
                "Fotografar peça final"
            ],
            "estimated_days": 2
        }
        self.repository.save(project)
        return project

    def create_project_from_image(self, title: str, description: str, image: Image.Image) -> SewingProject:
        project = SewingProject(
            title=title,
            description=description,
            source_type="image"
        )
        project.image_analysis = self.image_service.analyze(image)
        project.concepts = self.idea_service.generate_concepts(description or title)
        project.materials = self.materials_service.suggest_materials(project)
        project.costing = self.cost_service.calculate(project.materials, labor_hours=8.0)
        project.creative_assets = self.creative_service.generate_brand_assets(project)
        project.production_plan = {
            "checklist": [
                "Analisar referencia visual",
                "Definir proporcoes",
                "Escolher materiais",
                "Criar molde base",
                "Executar prototipo",
                "Ajustar estrutura",
                "Finalizar costura",
                "Validar funcionalidade"
            ],
            "estimated_days": 3
        }
        self.repository.save(project)
        return project


### Celula 14 - Dashboard simples

In [ ]:
import pandas as pd
import plotly.express as px


def build_dashboard_data(repo: SQLiteProjectRepository):
    rows = repo.list_projects()
    df = pd.DataFrame(rows, columns=["id", "title", "description", "source_type"])
    return df


def build_dashboard_chart(df: pd.DataFrame):
    if df.empty:
        return None
    chart = px.histogram(
        df,
        x="source_type",
        color="source_type",
        title="Distribuição de projetos por tipo de origem"
    )
    return chart


### Celula 15 - Interface Gradio + escolha do LLM (sem mock)

> Este notebook usa apenas resposta real de modelo. Se o provider/modelo estiver indisponivel, ele gera erro explicito.

Defina o provider antes de executar esta celula:
- `VIBE_LLM_PROVIDER=auto` (padrao: Colab usa Qwen local via Transformers; fora do Colab tenta Ollama)
- `VIBE_LLM_PROVIDER=qwen` para forcar Transformers local
- `VIBE_LLM_PROVIDER=ollama` para usar Ollama
- `VIBE_LLM_PROVIDER=gemini` ou `openai` para API
- `VIBE_LLM_MODEL` opcional para informar o nome do modelo

In [ ]:
import gradio as gr
import os
import numpy as np
import json
import traceback


db_path = PROJECT_ROOT / "data" / "projects.db"
repo = SQLiteProjectRepository(db_path=db_path)
rag_service = SimpleRAGService()

LLM_PROVIDER_NAME = os.getenv("VIBE_LLM_PROVIDER", "auto")
LLM_MODEL_NAME = os.getenv("VIBE_LLM_MODEL", "")

def init_orchestrator():
    llm_provider = get_default_llm_provider(
        provider_name=LLM_PROVIDER_NAME,
        model_name=(LLM_MODEL_NAME or None),
    )
    return VibeSewingOrchestrator(llm_provider=llm_provider, repository=repo), llm_provider


try:
    orchestrator, llm = init_orchestrator()
    print(f"Provider ativo: {llm.__class__.__name__}")
except Exception as exc:
    orchestrator, llm = None, None
    print(f"Falha na inicializacao do provider: {exc}")


CUSTOM_CSS = """
body {
    background: linear-gradient(180deg, #ffffff 0%, #f4f7fb 100%);
}
.gradio-container {
    max-width: 1400px !important;
}
.main-title {
    font-size: 36px;
    font-weight: 800;
    color: #2b2f42;
}
.sub-title {
    font-size: 16px;
    color: #596579;
}
.card {
    border-radius: 18px;
    border: 1px solid #e3e8f2;
    box-shadow: 0 10px 30px rgba(93, 107, 153, 0.12);
}
"""


def _ensure_runtime_ready():
    if orchestrator is None or llm is None:
        raise RuntimeError(
            "LLM indisponivel. Configure VIBE_LLM_PROVIDER e VIBE_LLM_MODEL corretamente "
            "(ex.: qwen no Colab, ollama local, openai/gemini com chave)."
        )


def _text_error_outputs(message: str):
    msg = str(message).strip() or "Erro desconhecido"
    error_json = json.dumps({"erro": msg}, ensure_ascii=False, indent=2)
    return (
        pd.DataFrame([{"erro": msg}]),
        {"erro": msg},
        pd.DataFrame([{"erro": msg}]),
        pd.DataFrame([{"erro": msg}]),
        msg,
        msg,
        msg,
        f"- {msg}",
        error_json,
    )


def _image_error_outputs(message: str):
    msg = str(message).strip() or "Erro desconhecido"
    error_json = json.dumps({"erro": msg}, ensure_ascii=False, indent=2)
    return (
        {"erro": msg},
        pd.DataFrame([{"erro": msg}]),
        pd.DataFrame([{"erro": msg}]),
        msg,
        msg,
        f"- {msg}",
        None,
        error_json,
    )


def run_text_pipeline(title: str, description: str):
    try:
        _ensure_runtime_ready()
    except Exception as exc:
        return _text_error_outputs(f"Runtime indisponivel: {exc}")

    if not title or not title.strip():
        return _text_error_outputs("Informe o nome do projeto.")
    if not description or not description.strip():
        return _text_error_outputs("Descreva sua ideia para gerar conceitos.")

    try:
        project = orchestrator.create_project_from_text(title=title.strip(), description=description.strip())

        materials_df = pd.DataFrame([{
            "nome": item.name,
            "categoria": item.category,
            "quantidade": item.quantity,
            "unidade": item.unit,
            "custo_unitario": item.estimated_unit_cost,
            "subtotal": item.subtotal,
        } for item in project.materials])

        checklist = "\n".join([f"- {item}" for item in project.production_plan["checklist"]])

        return (
            pd.DataFrame(project.concepts),
            project.image_analysis,
            materials_df,
            pd.DataFrame([project.costing]),
            project.creative_assets["story"],
            project.creative_assets["instagram"],
            project.creative_assets["linkedin"],
            checklist,
            project.to_json()
        )
    except Exception as exc:
        print("[run_text_pipeline]", traceback.format_exc())
        return _text_error_outputs(f"Falha ao gerar projeto por texto: {exc}")


def run_image_pipeline(title: str, description: str, image):
    try:
        _ensure_runtime_ready()
    except Exception as exc:
        return _image_error_outputs(f"Runtime indisponivel: {exc}")

    if image is None:
        return _image_error_outputs("Envie uma imagem para analise.")

    clean_title = (title or "Projeto por imagem").strip()
    clean_desc = (description or "").strip()

    try:
        image_array = np.asarray(image)
        pil_image = Image.fromarray(np.clip(image_array, 0, 255).astype("uint8"))
        project = orchestrator.create_project_from_image(title=clean_title, description=clean_desc, image=pil_image)

        materials_df = pd.DataFrame([{
            "nome": item.name,
            "categoria": item.category,
            "quantidade": item.quantity,
            "unidade": item.unit,
            "custo_unitario": item.estimated_unit_cost,
            "subtotal": item.subtotal,
        } for item in project.materials])

        fig = orchestrator.sketch_service.generate_sketch_views(project.title)
        checklist = "\n".join([f"- {item}" for item in project.production_plan["checklist"]])

        return (
            project.image_analysis,
            materials_df,
            pd.DataFrame([project.costing]),
            project.creative_assets["description"],
            project.creative_assets["image_prompt"],
            checklist,
            fig,
            project.to_json()
        )
    except Exception as exc:
        print("[run_image_pipeline]", traceback.format_exc())
        return _image_error_outputs(f"Falha ao gerar projeto por imagem: {exc}")


def run_rag_pipeline(knowledge_text: str, question: str):
    try:
        _ensure_runtime_ready()
    except Exception as exc:
        return f"Runtime indisponivel: {exc}", ""

    if not knowledge_text or not knowledge_text.strip():
        return "Informe um texto tecnico para criar a base RAG.", ""
    if not question or not question.strip():
        return "Informe uma pergunta tecnica.", ""

    try:
        rag_service.ingest_long_text(knowledge_text)
        result = rag_service.answer_with_context(question=question.strip(), llm_provider=llm, top_k=3)
        contexts_md = "\n\n".join([f"- {c}" for c in result["contexts"]])
        return result["answer"], contexts_md
    except Exception as exc:
        print("[run_rag_pipeline]", traceback.format_exc())
        return f"Falha no assistente tecnico: {exc}", ""


def refresh_dashboard():
    df = build_dashboard_data(repo)
    chart = build_dashboard_chart(df)
    return df, chart


with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft()) as demo:
    gr.HTML("""
    <div class="main-title">Vibe Sewing Lab</div>
    <div class="sub-title">AI-powered Creative Sewing Assistant</div>
    """)

    with gr.Tabs():
        with gr.Tab("Criador de Ideias"):
            with gr.Row():
                title_input = gr.Textbox(label="Nome do projeto", placeholder="Ex.: Bolsa de viagem Aurora")
                description_input = gr.Textbox(
                    label="Descreva sua ideia",
                    lines=5,
                    placeholder="Quero uma bolsa para viagem, resistente, elegante e com divisorias."
                )
            text_button = gr.Button("Gerar projeto completo", variant="primary")

            concepts_output = gr.Dataframe(label="Conceitos gerados")
            analysis_output = gr.JSON(label="Analise visual")
            materials_output = gr.Dataframe(label="Lista de materiais")
            costing_output = gr.Dataframe(label="Custos")
            story_output = gr.Textbox(label="Historia da bolsa", lines=8)
            insta_output = gr.Textbox(label="Instagram", lines=4)
            linkedin_output = gr.Textbox(label="LinkedIn", lines=4)
            checklist_output = gr.Textbox(label="Checklist de producao", lines=10)
            json_output = gr.Code(label="JSON do projeto", language="json")

            text_button.click(
                fn=run_text_pipeline,
                inputs=[title_input, description_input],
                outputs=[
                    concepts_output,
                    analysis_output,
                    materials_output,
                    costing_output,
                    story_output,
                    insta_output,
                    linkedin_output,
                    checklist_output,
                    json_output,
                ]
            )

        with gr.Tab("Analise por Foto"):
            image_title = gr.Textbox(label="Nome do projeto")
            image_description = gr.Textbox(label="Descricao complementar", lines=4)
            image_input = gr.Image(label="Envie a foto", type="numpy")
            image_button = gr.Button("Analisar imagem e gerar projeto", variant="primary")

            image_analysis_output = gr.JSON(label="Resultado da analise")
            image_materials_output = gr.Dataframe(label="Materiais")
            image_costing_output = gr.Dataframe(label="Precificacao")
            image_desc_output = gr.Textbox(label="Descricao comercial", lines=5)
            image_prompt_output = gr.Textbox(label="Prompt de imagem", lines=4)
            image_checklist_output = gr.Textbox(label="Checklist", lines=8)
            image_sketch_output = gr.Plot(label="Sketch tecnico")
            image_json_output = gr.Code(label="JSON do projeto", language="json")

            image_button.click(
                fn=run_image_pipeline,
                inputs=[image_title, image_description, image_input],
                outputs=[
                    image_analysis_output,
                    image_materials_output,
                    image_costing_output,
                    image_desc_output,
                    image_prompt_output,
                    image_checklist_output,
                    image_sketch_output,
                    image_json_output,
                ]
            )

        with gr.Tab("Assistente Tecnico (RAG)"):
            rag_context_input = gr.Textbox(
                label="Base tecnica (cole aqui instrucoes, boas praticas, materiais)",
                lines=10,
                placeholder="Ex.: orientacoes de reforco de alca, tipos de entretela, ordem de montagem..."
            )
            rag_question_input = gr.Textbox(
                label="Pergunta tecnica",
                lines=3,
                placeholder="Como reforcar a alca para bolsa de viagem pesada?"
            )
            rag_button = gr.Button("Consultar base tecnica", variant="primary")
            rag_answer_output = gr.Textbox(label="Resposta tecnica", lines=10)
            rag_contexts_output = gr.Markdown(label="Trechos recuperados")

            rag_button.click(
                fn=run_rag_pipeline,
                inputs=[rag_context_input, rag_question_input],
                outputs=[rag_answer_output, rag_contexts_output],
            )

        with gr.Tab("Dashboard"):
            dash_button = gr.Button("Atualizar dashboard")
            dash_table = gr.Dataframe(label="Projetos salvos")
            dash_chart = gr.Plot(label="Grafico")
            dash_button.click(
                fn=refresh_dashboard,
                inputs=[],
                outputs=[dash_table, dash_chart]
            )

try:
    demo.launch(share=True, debug=True, show_error=True)
except Exception:
    demo.launch(share=False, debug=True, show_error=True)

### Celula 16 - Exportar para Python e ZIP

> Execute esta celula apos validar a execucao para gerar um `.py` pronto para repositorio GitHub.

In [ ]:
import json
from pathlib import Path
from zipfile import ZipFile


def find_notebook_path() -> Path:
    candidates = [
        Path("/content/VibeSewing.ipynb"),
        Path("VibeSewing.ipynb"),
        Path.cwd() / "VibeSewing.ipynb",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Nao encontrei VibeSewing.ipynb no ambiente atual.")


def export_notebook_to_py(notebook_path: Path, output_py: Path) -> Path:
    raw = notebook_path.read_text(encoding="utf-8")
    nb_data = json.loads(raw)

    code_blocks = []
    for cell in nb_data.get("cells", []):
        if cell.get("cell_type") == "code":
            lines = cell.get("source", [])
            normalized = []
            for line in lines:
                normalized.append(line if line.endswith("\n") else f"{line}\n")
            code_blocks.append("".join(normalized).rstrip())

    script_header = [
        "# Auto-gerado a partir do notebook VibeSewing.ipynb",
        "# Revise os comandos !pip conforme o ambiente de destino.",
        "",
    ]
    output_text = "\n".join(script_header) + "\n\n# %%\n\n" + "\n\n# %%\n\n".join(code_blocks) + "\n"
    output_py.write_text(output_text, encoding="utf-8")
    return output_py


notebook_path = find_notebook_path()
export_dir = PROJECT_ROOT / "data" / "exports"
export_dir.mkdir(parents=True, exist_ok=True)

py_path = export_notebook_to_py(notebook_path, export_dir / "vibe_sewing_lab.py")
zip_path = export_dir / "vibe_sewing_lab_export.zip"

with ZipFile(zip_path, "w") as zf:
    zf.write(py_path, arcname=py_path.name)
    zf.write(notebook_path, arcname=notebook_path.name)

print(f"Script Python exportado: {py_path}")
print(f"Pacote ZIP gerado: {zip_path}")